In [1]:
import pandas as pd
from pathlib import Path
import os


VIDEO_ROOT = Path("/kaggle/input/datasets/indiff/videos")  
CSV_PATH = Path("/kaggle/input/datasets/indiff/labels/bah-video.csv")


df = pd.read_csv(CSV_PATH)


df["full_path"] = df["video-path"].apply(
    lambda x: str(VIDEO_ROOT / str(x).strip())
)

print("Проверка путей:\n")

for p in df["full_path"].sample(5):
    print(p)
    print("Exists:", os.path.exists(p))
    print("-" * 60)


Проверка путей:

/kaggle/input/datasets/indiff/videos/Videos/82744/Visite_1/82744_Question_4_2024-11-27_20-57-33_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82614/Visite_1/82614_Question_1_2024-11-05_13-28-19_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82684/Visite_1/82684_Question_2_2024-11-14_10-51-40_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82994/Visite_1/82994_Question_4_2025-04-28_13-16-08_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82820/Visite_1/82820_Question_6_2025-02-02_05-04-51_Video.mp4
Exists: True
------------------------------------------------------------


In [2]:
print(df["video-path"].iloc[0])
print(df["full_path"].iloc[0])


Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4
/kaggle/input/datasets/indiff/videos/Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4


In [3]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))


998 214 215


In [4]:
pip install decord transformers accelerate timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 95.8 MB/s eta 0:00:00:00:01:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from transformers import VideoMAEForVideoClassification
from sklearn.metrics import accuracy_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from decord import VideoReader, cpu
from tqdm import tqdm
import random

train_transform = v2.Compose([
    v2.Resize((256, 256), antialias=True),
    v2.RandomResizedCrop(224, scale=(0.8, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5), 
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class RobustVideoDataset(Dataset):
    def __init__(self, dataframe, num_frames=16, transform=None, is_train=False):
        self.df = dataframe.reset_index(drop=True)
        self.num_frames = num_frames
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def load_video(self, path):
        vr = VideoReader(path, ctx=cpu(0))
        total_frames = len(vr)
        

        if self.is_train and total_frames > self.num_frames:

            max_offset = (total_frames - 1) // self.num_frames
            start = random.randint(0, max_offset) if max_offset > 0 else 0
            indices = np.linspace(start, total_frames - 1, self.num_frames).astype(int)
        else:

            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)

        frames = vr.get_batch(indices).asnumpy() # (T, H, W, C)
        
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        return frames

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video = self.load_video(row["full_path"])

        if self.transform:
            video = self.transform(video) # v2 применяет трансформации ко всем кадрам одинаково

        label = torch.tensor(row["label"]).long()
        return video, label

train_dataset = RobustVideoDataset(train_df, transform=train_transform, is_train=True)
val_dataset = RobustVideoDataset(val_df, transform=val_test_transform, is_train=False)
test_dataset = RobustVideoDataset(test_df, transform=val_test_transform, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, num_workers=4, pin_memory=True)

2026-02-22 19:19:54.997104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771787995.190289      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771787995.242113      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771787995.717065      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771787995.717113      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771787995.717116      55 computation_placer.cc:177] computation placer alr

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",
    num_labels=2,
    ignore_mismatched_sizes=True
)

# ЗАМОРОЗКА: Морозим первые 8 слоев трансформера из 12, чтобы избежать переобучения
for param in model.videomae.encoder.layer[:8].parameters():
    param.requires_grad = False

model.to(device)

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/377M [00:00<?, ?B/s]

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VideoMAEForVideoClassification(
  (videomae): VideoMAEModel(
    (embeddings): VideoMAEEmbeddings(
      (patch_embeddings): VideoMAEPatchEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (encoder): VideoMAEEncoder(
      (layer): ModuleList(
        (0-11): 12 x VideoMAELayer(
          (attention): VideoMAEAttention(
            (attention): VideoMAESelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): VideoMAESelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): VideoMAEIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
    

In [7]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=15) # Плавно снижаем LR

In [8]:
def train_epoch():
    model.train()
    total_loss = 0
    for videos, labels in tqdm(train_loader, desc="Training"):
        videos, labels = videos.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos).logits
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    scheduler.step()
    return total_loss / len(train_loader)

In [9]:
def evaluate(loader, desc="Evaluating"):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for videos, labels in tqdm(loader, desc=desc):
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos).logits
            predicted = torch.argmax(outputs, dim=1)
            
            preds.extend(predicted.cpu().numpy())
            true.extend(labels.cpu().numpy())
            
    acc = accuracy_score(true, preds)
    mf1 = f1_score(true, preds, average='macro')
    return acc, mf1

In [10]:
best_val_mf1 = 0.0

for epoch in range(15):
    print(f"\n--- Epoch {epoch+1}/15 ---")
    loss = train_epoch()
    val_acc, val_mf1 = evaluate(val_loader, desc="Validation")
    
    print(f"Loss: {loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_mf1:.4f}")
    
    if val_mf1 > best_val_mf1:
        best_val_mf1 = val_mf1
        torch.save(model.state_dict(), 'best_model.pth')
        print("-> Модель улучшилась, сохраняем!")


--- Epoch 1/15 ---


Validation: 100%|██████████| 54/54 [01:20<00:00,  1.49s/it]


Loss: 0.7115 | Val Acc: 0.5467 | Val Macro F1: 0.3535
-> Модель улучшилась, сохраняем!

--- Epoch 2/15 ---


Validation: 100%|██████████| 54/54 [01:20<00:00,  1.49s/it]


Loss: 0.7003 | Val Acc: 0.5327 | Val Macro F1: 0.3882
-> Модель улучшилась, сохраняем!

--- Epoch 3/15 ---


...

Loss: 0.5639 | Val Acc: 0.5561 | Val Macro F1: 0.5439

--- Epoch 14/15 ---


Validation: 100%|██████████| 54/54 [01:19<00:00,  1.46s/it]


Loss: 0.5561 | Val Acc: 0.5607 | Val Macro F1: 0.5464

--- Epoch 15/15 ---


Validation: 100%|██████████| 54/54 [01:18<00:00,  1.46s/it]

Loss: 0.5568 | Val Acc: 0.5701 | Val Macro F1: 0.5615


In [12]:
print("\n=== Оценка на тестовых данных ===")
model.load_state_dict(torch.load('best_model.pth'))
test_acc, test_mf1 = evaluate(test_loader, desc="Testing")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_mf1:.4f}")


=== Оценка на тестовых данных ===


Testing: 100%|██████████| 54/54 [01:15<00:00,  1.39s/it]

Test Accuracy: 0.6093
Test Macro F1: 0.6079
